In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install ultralytics
# !mkdir -p /kaggle/working/frames
# !ffmpeg -i "/kaggle/input/datasets/aryannaik225/dubai-mall-walking-tour/Dubai Mall The Largest Shopping Mall in the World Walking Tour 4K.mp4" -t 3600 -vf "fps=1,scale=-1:720" -q:v 2 /kaggle/working/frames/frame_%04d.jpg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00a 0:00:01


In [4]:
import cv2
import os
import glob
from ultralytics import YOLO
from tqdm.notebook import tqdm

frames_dir = '/kaggle/working/frames'
output_dir = '/kaggle/working/extracted_people'

# Re-create the empty directory
os.makedirs(output_dir, exist_ok=True)

print("🚀 Loading YOLO Engine...")
model = YOLO('yolo26n.pt') 

# Get all the images FFmpeg just ripped
frame_files = sorted(glob.glob(os.path.join(frames_dir, '*.jpg')))
print(f"📁 Found {len(frame_files)} keyframes to process.")

person_count = 0

for frame_path in tqdm(frame_files, desc="Scanning Keyframes"):
    frame = cv2.imread(frame_path)
    if frame is None:
        continue
        
    # GPU inference!
    results = model(frame, classes=[0], verbose=False, device=0)

    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0].item()
            
            height = y2 - y1
            width = x2 - x1
            
            # Filter out low-confidence or tiny blurry background people
            if confidence > 0.60 and height > 50 and width > 20:
                cropped_person = frame[y1:y2, x1:x2]
                
                save_path = os.path.join(output_dir, f"person_{person_count}.jpg")
                cv2.imwrite(save_path, cropped_person)
                person_count += 1

print("\n" + "="*40)
print(f"🎉 UNCAPPED EXTRACTION COMPLETE!")
print(f"📸 Total unique people cropped and saved: {person_count}")
print("="*40)

🚀 Loading YOLO Engine...
📁 Found 3600 keyframes to process.


Scanning Keyframes:   0%|          | 0/3600 [00:00<?, ?it/s]


🎉 UNCAPPED EXTRACTION COMPLETE!
📸 Total unique people cropped and saved: 12577


In [10]:
!rm -rf /kaggle/working/.virtual_documents
!rm -rf /kaggle/working/processed_dataset
# !rm -rf /kaggle/working/720p_mall.mp4

In [5]:
!pip install transformers

In [12]:
# Zip the entire folder into one clean file
!zip -r -q /kaggle/working/processed_dataset.zip /kaggle/working/processed_dataset

In [ ]:
from IPython.display import FileLink

# This creates a clickable link to download your zip file
FileLink(r'extracted_people.zip')

In [12]:
import torch
import os
import glob
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm.notebook import tqdm

output_dir = '/kaggle/working/extracted_people'
images = sorted(glob.glob(os.path.join(output_dir, '*.jpg')))

print("🚀 Loading Strict CLIP VLM...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# THE FIX 1: Semantic Prompts (CLIP understands concepts, not numbers)
gender_prompts = [
    "a clear photo of a man", 
    "a clear photo of a woman"
]

age_prompts = [
    "a photo of a young child or kid",               # 0 (0-15)
    "a photo of a teenager or college student",      # 1 (16-30)
    "a photo of a middle-aged adult",                # 2 (31-45)
    "a photo of an older adult with gray hair",      # 3 (46-60)
    "a photo of an elderly senior citizen"           # 4 (60+)
]

results = []
LIMIT = 500 # Let's test 100 again

# THE FIX 2: The Confidence Kill-Switch
CONFIDENCE_THRESHOLD = 0.60 

print(f"🗂️ Testing {LIMIT} images with a strict {CONFIDENCE_THRESHOLD*100}% confidence filter...")

with torch.no_grad():
    for img_path in tqdm(images[:LIMIT], desc="Strict Auto-Labeling"):
        try:
            image = Image.open(img_path).convert("RGB")
            
            # --- GENDER ---
            inputs_gender = processor(text=gender_prompts, images=image, return_tensors="pt", padding=True).to(device)
            probs_gender = model(**inputs_gender).logits_per_image.softmax(dim=1)[0]
            
            gender_confidence, gender_id = torch.max(probs_gender, dim=0)
            
            # --- AGE ---
            inputs_age = processor(text=age_prompts, images=image, return_tensors="pt", padding=True).to(device)
            probs_age = model(**inputs_age).logits_per_image.softmax(dim=1)[0]
            
            age_confidence, age_id = torch.max(probs_age, dim=0)
            
            # 🛑 THE FILTER: Only save it if the AI is highly confident on BOTH
            if gender_confidence.item() >= CONFIDENCE_THRESHOLD and age_confidence.item() >= CONFIDENCE_THRESHOLD:
                filename = os.path.basename(img_path)
                results.append({
                    "image_path": filename,
                    "gender": gender_id.item(),
                    "age_range": age_id.item(),
                    "gender_conf": round(gender_confidence.item(), 2),
                    "age_conf": round(age_confidence.item(), 2)
                })
            
        except Exception as e:
            continue

df = pd.DataFrame(results)
df.to_csv('/kaggle/working/strict_teacher_labels.csv', index=False)

print("\n" + "="*50)
print(f"✅ STRICT VLM COMPLETE!")
print(f"📉 Out of {LIMIT} images, CLIP was highly confident on {len(df)} images.")
print("="*50)

if len(df) > 0:
    print(df.head(10))
else:
    print("⚠️ No images passed the 85% confidence threshold! We might need to lower it.")

🚀 Loading Strict CLIP VLM...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🗂️ Testing 500 images with a strict 60.0% confidence filter...


Strict Auto-Labeling:   0%|          | 0/500 [00:00<?, ?it/s]


✅ STRICT VLM COMPLETE!
📉 Out of 500 images, CLIP was highly confident on 140 images.
         image_path  gender  age_range  gender_conf  age_conf
0      person_0.jpg       0          2         0.94      0.81
1     person_10.jpg       0          2         0.94      0.83
2  person_10001.jpg       0          2         0.92      0.88
3  person_10002.jpg       0          2         0.81      0.78
4  person_10009.jpg       0          2         0.95      0.70
5   person_1001.jpg       0          2         0.93      0.62
6  person_10012.jpg       0          2         0.91      0.78
7  person_10014.jpg       0          1         0.78      0.65
8  person_10017.jpg       1          2         0.98      0.63
9  person_10022.jpg       1          2         0.97      0.63


In [13]:
print("📊 GENDER DISTRIBUTION (0=Male, 1=Female):")
print(df['gender'].value_counts())

print("\n📊 AGE DISTRIBUTION (0=Kids, 1=Young, 2=Adult, 3=Mid, 4=Senior):")
print(df['age_range'].value_counts())

📊 GENDER DISTRIBUTION (0=Male, 1=Female):
gender
1    73
0    67
Name: count, dtype: int64

📊 AGE DISTRIBUTION (0=Kids, 1=Young, 2=Adult, 3=Mid, 4=Senior):
age_range
2    118
4     13
1      5
0      4
Name: count, dtype: int64


# TRYING SOMETHING

In [3]:
# This creates a lightweight 720p version of the video so your CPU doesn't die during tracking
!ffmpeg -i "/kaggle/input/datasets/aryannaik225/dubai-mall-walking-tour/Dubai Mall The Largest Shopping Mall in the World Walking Tour 4K.mp4" -t 3600 -vf scale=-1:720 -c:v libx264 -preset veryfast /kaggle/working/720p_mall.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [6]:
import cv2
import os
from ultralytics import YOLO
from tqdm.notebook import tqdm

# Point this to your 1080p file!
video_path = '/kaggle/input/datasets/aryannaik225/dubai-mall-walking-tour/Dubai Mall The Largest Shopping Mall in the World Walking Tour 4K.mp4' 
output_dir = '/kaggle/working/unique_people'
os.makedirs(output_dir, exist_ok=True)

print("🚀 Loading YOLO Engine...")
model = YOLO('yolo26n.pt')

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# --- THE SPEED UPGRADES ---
MAX_FRAMES_TO_PROCESS = 216000  # Stop after ~27 minutes of footage. 
FRAME_SKIP = 2                  # Process every 2nd frame to double the speed!

saved_track_ids = set()
saved_count = 0
frames_processed = 0

for _ in tqdm(range(total_frames), desc="Tracking Unique People"):
    ret, frame = cap.read()
    if not ret:
        break

    frames_processed += 1
    
    # Trigger the Kill-Switch
    if frames_processed > MAX_FRAMES_TO_PROCESS:
        print(f"\n🛑 Hit our {MAX_FRAMES_TO_PROCESS} frame limit! We have plenty of data.")
        break

    # Trigger the Frame Skipper (Instantly halves processing time)
    if frames_processed % FRAME_SKIP != 0:
        continue

    # Shrink the frame in RAM
    frame = cv2.resize(frame, (1280, 720))

    # BoT-SORT Tracking
    results = model.track(frame, persist=True, classes=[0], tracker="botsort.yaml", verbose=False, device=0)
    
    if results[0].boxes is None or results[0].boxes.id is None:
        continue
        
    boxes = results[0].boxes.xyxy.cpu().numpy()
    track_ids = results[0].boxes.id.int().cpu().tolist()
    confidences = results[0].boxes.conf.cpu().numpy()
    
    for box, track_id, conf in zip(boxes, track_ids, confidences):
        # The Vault: Only save them if we haven't seen this ID before
        if track_id not in saved_track_ids:
            x1, y1, x2, y2 = map(int, box)
            height, width = y2 - y1, x2 - x1
            
            if conf > 0.60 and height > 80 and width > 40:
                cropped_person = frame[y1:y2, x1:x2]
                
                save_path = os.path.join(output_dir, f"person_id_{track_id}.jpg")
                cv2.imwrite(save_path, cropped_person)
                
                saved_track_ids.add(track_id)
                saved_count += 1

cap.release()
print("\n" + "="*50)
print(f"🎉 TRACKING EXTRACTION COMPLETE!")
print(f"📸 Total GUARANTEED UNIQUE people saved: {saved_count}")
print("="*50)

🚀 Loading YOLO Engine...


Tracking Unique People:   0%|          | 0/809832 [00:00<?, ?it/s]

WARNING ⚠️ not enough matching points

🛑 Hit our 216000 frame limit! We have plenty of data.

🎉 TRACKING EXTRACTION COMPLETE!
📸 Total GUARANTEED UNIQUE people saved: 5612


In [10]:
# Install DeepFace just in case the runtime reset
!pip install deepface tf-keras -q

In [11]:
import os
import glob
import pandas as pd
from deepface import DeepFace
from tqdm.notebook import tqdm

output_dir = '/kaggle/working/unique_people'
images = sorted(glob.glob(os.path.join(output_dir, '*.jpg')))

results = []

print(f"🗂️ Firing up DeepFace to scan all {len(images)} UNIQUE images...")

for img_path in tqdm(images, desc="DeepFace Scanning & Labeling"):
    try:
        # enforce_detection=True is our Bouncer! (Throws out back-of-heads)
        analysis = DeepFace.analyze(img_path, 
                                    actions=['age', 'gender'], 
                                    enforce_detection=True, 
                                    silent=True)
        
        face_data = analysis[0]
        exact_age = face_data['age']
        dominant_gender = face_data['dominant_gender'] 
        
        # --- MAP GENDER (0=Male, 1=Female) ---
        gender_id = 0 if dominant_gender == 'Man' else 1
        
        # --- MAP AGE (0-4) ---
        if exact_age <= 15:
            age_id = 0
        elif 16 <= exact_age <= 30:
            age_id = 1
        elif 31 <= exact_age <= 45:
            age_id = 2
        elif 46 <= exact_age <= 60:
            age_id = 3
        else:
            age_id = 4
            
        filename = os.path.basename(img_path)
        results.append({
            "image_path": filename,
            "gender": gender_id,
            "age_range": age_id
        })
        
    except ValueError:
        # Bouncer triggered: No face found. Skip it!
        continue
    except Exception as e:
        continue

# Save the final pristine dataset
df_deepface = pd.DataFrame(results)
df_deepface.to_csv('/kaggle/working/final_labels.csv', index=False)

print("\n" + "="*50)
print(f"✅ DEEPFACE PIPELINE COMPLETE!")
print(f"📉 Out of {len(images)} crops, DeepFace kept {len(df_deepface)} pristine, front-facing images.")
print("="*50)

if len(df_deepface) > 0:
    print("\n📊 FINAL GENDER DISTRIBUTION:")
    print(df_deepface['gender'].value_counts())
    print("\n📊 FINAL AGE DISTRIBUTION (0=Kids, 1=Young, 2=Adult, 3=Mid, 4=Senior):")
    print(df_deepface['age_range'].value_counts().sort_index())

26-03-06 07:37:57 - Directory /root/.deepface has been created
26-03-06 07:37:57 - Directory /root/.deepface/weights has been created
🗂️ Firing up DeepFace to scan all 5612 UNIQUE images...


DeepFace Scanning & Labeling:   0%|          | 0/5612 [00:00<?, ?it/s]

I0000 00:00:1772782678.169100      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13505 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


26-03-06 07:37:58 - 🔗 age_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5 to /root/.deepface/weights/age_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5
To: /root/.deepface/weights/age_model_weights.h5

  0%|          | 0.00/539M [00:00<?, ?B/s]
  2%|▏         | 9.44M/539M [00:00<00:05, 93.6MB/s]
  7%|▋         | 36.2M/539M [00:00<00:02, 193MB/s] 
 12%|█▏        | 65.5M/539M [00:00<00:02, 236MB/s]
 18%|█▊        | 94.9M/539M [00:00<00:01, 255MB/s]
 23%|██▎       | 124M/539M [00:00<00:01, 266MB/s] 
 28%|██▊       | 153M/539M [00:00<00:01, 273MB/s]
 34%|███▍      | 182M/539M [00:00<00:01, 275MB/s]
 39%|███▉      | 212M/539M [00:00<00:01, 278MB/s]
 44%|████▍     | 240M/539M [00:00<00:01, 256MB/s]
 50%|████▉     | 267M/539M [00:01<00:01, 261MB/s]
 55%|█████▍    | 296M/539M [00:01<00:00, 267MB/s]
 60%|██████    | 325M/539M [00:01<00:00, 273MB/s]
 66%|██████▌   | 354M/539M [00:01<00:00, 276MB/s]
 71%|███████   | 382M/539M [00:01<00:00, 257MB/s]
 76%|███████▋  | 411M/539M [00:01<00:00, 265MB/s]
 82%|████████▏ | 440M/539M [00:01<00:00, 

26-03-06 07:38:04 - 🔗 gender_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5 to /root/.deepface/weights/gender_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5
To: /root/.deepface/weights/gender_model_weights.h5

  0%|          | 0.00/537M [00:00<?, ?B/s]
  2%|▏         | 11.0M/537M [00:00<00:05, 105MB/s]
  7%|▋         | 38.8M/537M [00:00<00:02, 204MB/s]
 13%|█▎        | 68.7M/537M [00:00<00:01, 247MB/s]
 17%|█▋        | 93.8M/537M [00:00<00:01, 242MB/s]
 23%|██▎       | 126M/537M [00:00<00:01, 269MB/s] 
 29%|██▊       | 154M/537M [00:00<00:01, 273MB/s]
 35%|███▍      | 186M/537M [00:00<00:01, 287MB/s]
 40%|████      | 215M/537M [00:00<00:01, 279MB/s]
 46%|████▌     | 246M/537M [00:00<00:01, 288MB/s]
 51%|█████▏    | 276M/537M [00:01<00:00, 289MB/s]
 57%|█████▋    | 306M/537M [00:01<00:00, 293MB/s]
 63%|██████▎   | 336M/537M [00:01<00:00, 291MB/s]
 68%|██████▊   | 368M/537M [00:01<00:00, 288MB/s]
 74%|███████▍  | 399M/537M [00:01<00:00, 291MB/s]
 80%|████████  | 430M/537M [00:01<00:00, 294MB/s]
 86%|████████▌ | 461M/537M [00:01<00:


✅ DEEPFACE PIPELINE COMPLETE!
📉 Out of 5612 crops, DeepFace kept 150 pristine, front-facing images.

📊 FINAL GENDER DISTRIBUTION:
gender
0    122
1     28
Name: count, dtype: int64

📊 FINAL AGE DISTRIBUTION (0=Kids, 1=Young, 2=Adult, 3=Mid, 4=Senior):
age_range
1    78
2    72
Name: count, dtype: int64


In [12]:
import torch
import os
import glob
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm.notebook import tqdm

output_dir = '/kaggle/working/unique_people'
images = sorted(glob.glob(os.path.join(output_dir, '*.jpg')))

print("🚀 Loading Gender-Only CLIP VLM...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Semantic prompts for the entire body, not just the face
gender_prompts = [
    "a photo of a man, male body", 
    "a photo of a woman, female body"
]

results = []
CONFIDENCE_THRESHOLD = 0.75 

print(f"🗂️ Scanning {len(images)} unique people for Gender...")

with torch.no_grad():
    for img_path in tqdm(images, desc="Extracting Gender"):
        try:
            image = Image.open(img_path).convert("RGB")
            
            inputs = processor(text=gender_prompts, images=image, return_tensors="pt", padding=True).to(device)
            probs = model(**inputs).logits_per_image.softmax(dim=1)[0]
            
            confidence, gender_id = torch.max(probs, dim=0)
            
            # If the AI is at least 75% sure of the gender, keep it!
            if confidence.item() >= CONFIDENCE_THRESHOLD:
                filename = os.path.basename(img_path)
                results.append({
                    "image_path": filename,
                    "gender": gender_id.item(),
                    "confidence": round(confidence.item(), 2)
                })
            
        except Exception as e:
            continue

df_gender = pd.DataFrame(results)
df_gender.to_csv('/kaggle/working/gender_labels.csv', index=False)

print("\n" + "="*50)
print(f"✅ GENDER EXTRACTION COMPLETE!")
print(f"📉 Out of {len(images)} images, CLIP was highly confident on {len(df_gender)} images.")
print("="*50)

if len(df_gender) > 0:
    print("\n📊 FINAL GENDER DISTRIBUTION (0=Male, 1=Female):")
    print(df_gender['gender'].value_counts())

🚀 Loading Gender-Only CLIP VLM...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

🗂️ Scanning 5612 unique people for Gender...


Extracting Gender:   0%|          | 0/5612 [00:00<?, ?it/s]


✅ GENDER EXTRACTION COMPLETE!
📉 Out of 5612 images, CLIP was highly confident on 2620 images.

📊 FINAL GENDER DISTRIBUTION (0=Male, 1=Female):
gender
0    1884
1     736
Name: count, dtype: int64


In [13]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/working/gender_labels.csv')

# 1. Separate the classes
males = df[df['gender'] == 0]
females = df[df['gender'] == 1]

# 2. Undersample males to perfectly match the female count (736)
min_class_size = len(females)
males_sampled = males.sample(n=min_class_size, random_state=42)

# Combine them into a perfectly balanced 50/50 dataset
balanced_df = pd.concat([males_sampled, females]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"⚖️ Dataset balanced! Total images: {len(balanced_df)} ({min_class_size} Male, {min_class_size} Female)")

# 3. Split 80% for Training, 20% for Validation (Testing)
train_df, val_df = train_test_split(balanced_df, test_size=0.2, random_state=42, stratify=balanced_df['gender'])

# 4. Create the PyTorch directory structure
base_dir = '/kaggle/working/swin_dataset'
for split in ['train', 'val']:
    for gender in ['male', 'female']:
        os.makedirs(os.path.join(base_dir, split, gender), exist_ok=True)

# 5. Copy the images into their new homes
source_dir = '/kaggle/working/unique_people'

def copy_images(dataframe, split_name):
    count = 0
    for _, row in dataframe.iterrows():
        src = os.path.join(source_dir, row['image_path'])
        gender_folder = 'male' if row['gender'] == 0 else 'female'
        dst = os.path.join(base_dir, split_name, gender_folder, row['image_path'])
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            count += 1
    return count

print("📂 Copying images into PyTorch folder structure...")
train_count = copy_images(train_df, 'train')
val_count = copy_images(val_df, 'val')

print("\n" + "="*50)
print("✅ SWIN DATASET READY FOR TRAINING!")
print(f"🏋️ Training Images: {train_count}")
print(f"🧪 Validation Images: {val_count}")
print("="*50)

⚖️ Dataset balanced! Total images: 1472 (736 Male, 736 Female)
📂 Copying images into PyTorch folder structure...

✅ SWIN DATASET READY FOR TRAINING!
🏋️ Training Images: 1177
🧪 Validation Images: 295


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time

# 1. Hyperparameters
BATCH_SIZE = 64
EPOCHS = 5 # Let's do 5 quick epochs to see how fast it learns!
LEARNING_RATE = 3e-5 # Transformers like small learning rates
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()}...")

# 2. Data Augmentation (Teaching the AI to handle weird angles and lighting)
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # Zoom in randomly
    transforms.RandomHorizontalFlip(),                   # Flip images left/right
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Change lighting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # No random zooming for the test!
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load the Datasets using our new folder structure
base_dir = '/kaggle/working/swin_dataset'
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# 4. Load the Swin Transformer and modify the head
model = models.swin_t(weights='DEFAULT')

# Surgical replacement of the final layer
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) # Only 2 outputs: Male or Female
model = model.to(DEVICE)

# 5. The Engine: Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# 6. The Training Loop
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    # --- TRAINING PHASE ---
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Save the absolute best version of the model
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 1.0 to your Kaggle Working Directory!
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_gender.pth')
print("💾 Saved Version 1.0 weights as 'swin_v1_gender.pth'")

🚀 Firing up PyTorch on CUDA...
🗂️ Classes detected by PyTorch: {'female': 0, 'male': 1}
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 203MB/s]  



Epoch 1/5
---------------


Training:   0%|          | 0/19 [00:00<?, ?it/s]

Testing:   0%|          | 0/5 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/5 [00:00<?, ?it/s]

Train Loss: 0.3944 | Train Acc: 0.8352
Val Loss:   0.5044 | Val Acc:   0.7729

Epoch 5/5
---------------


Training:   0%|          | 0/19 [00:00<?, ?it/s]

Testing:   0%|          | 0/5 [00:00<?, ?it/s]

Train Loss: 0.3644 | Train Acc: 0.8488
Val Loss:   0.5655 | Val Acc:   0.7458

🎉 TRAINING COMPLETE in 1m 30s
🏆 Best Validation Accuracy: 0.7729
💾 Saved Version 1.0 weights as 'swin_v1_gender.pth'


# Manually Labeled Dataset Code Below

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os

# 1. Hyperparameters
BATCH_SIZE = 32
EPOCHS = 10  # Doubled the training time for the Golden Dataset!
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Version 1.1...")

# 2. Data Augmentation
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load the Golden Dataset
base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000'
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# 4. Load the Swin Transformer
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) 
model = model.to(DEVICE)

# 5. The Engine
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# 6. The Training Loop
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 VERSION 1.1 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 1.1!
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_1_gender.pth')
print("💾 Saved Version 1.1 weights as 'swin_v1_1_gender.pth'")

🚀 Firing up PyTorch on CUDA for Version 1.1...
🗂️ Classes detected by PyTorch: {'female': 0, 'male': 1}
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 228MB/s] 



Epoch 1/10
---------------


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Testing:   0%|          | 0/8 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/8 [00:00<?, ?it/s]

Train Loss: 0.1236 | Train Acc: 0.9551
Val Loss:   0.8458 | Val Acc:   0.7460

Epoch 10/10
---------------


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Testing:   0%|          | 0/8 [00:00<?, ?it/s]

Train Loss: 0.1452 | Train Acc: 0.9461
Val Loss:   0.6913 | Val Acc:   0.8016

🎉 VERSION 1.1 TRAINING COMPLETE in 2m 33s
🏆 Best Validation Accuracy: 0.8016
💾 Saved Version 1.1 weights as 'swin_v1_1_gender.pth'


In [4]:
!pip install rembg onnxruntime opencv-python Pillow==9.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.9 MB/s eta 0:00:00:00:0100:01


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from rembg import remove
import requests

# --- 1. THE PURE NUMPY PREPROCESSING ENGINE ---
def enhance_and_remove_bg(input_path, output_path):
    try:
        # 1. Load Image via OpenCV
        img = cv2.imread(input_path)
        if img is None: return

        # 2. CLAHE (Lighting enhancement)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l_channel, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l_channel)
        merged = cv2.merge((cl, a, b))
        enhanced_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

        # 3. Convert to RGB for the Neural Network
        enhanced_rgb = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB)

        # 4. Background Removal (rembg takes numpy, returns RGBA numpy)
        no_bg_rgba = remove(enhanced_rgb)

        # 5. Extract the Math Channels
        rgb = no_bg_rgba[:, :, :3]
        alpha = no_bg_rgba[:, :, 3]

        # 6. Create a Pitch Black background array
        black_bg = np.zeros_like(rgb)
        
        # 7. Stamp the person onto the black background using the Alpha mask
        mask = alpha > 10 # 10 is a threshold to prevent fuzzy gray ghosting
        black_bg[mask] = rgb[mask]

        # 8. Convert back to BGR and Save
        final_bgr = cv2.cvtColor(black_bg, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, final_bgr)
        
    except Exception as e:
        print(f"Skipping corrupted image {input_path}: {e}")

# PATHS
raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/clean_v1_2_dataset'
processed_base_dir = '/kaggle/working/processed_dataset'

print("✨ Phase 1: Applying CLAHE + Background Removal (NumPy Engine)...")
for split in ['train', 'val']:
    for gender in ['male', 'female']:
        in_folder = os.path.join(raw_base_dir, split, gender)
        out_folder = os.path.join(processed_base_dir, split, gender)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{gender}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                
                # Check if already processed (saves time if you restart the cell)
                if not os.path.exists(out_path):
                    enhance_and_remove_bg(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. HYPERPARAMETERS ---
BATCH_SIZE = 32
EPOCHS = 10  
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Version 1.3...")

def custom_opencv_loader(path):
    """Bypasses Pillow's broken JPEG parser by forcing OpenCV to read the images."""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 4. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 VERSION 1.3 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 1.3
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_3_gender.pth')
print("💾 Saved Version 1.3 weights as 'swin_v1_3_gender.pth'")

✨ Phase 1: Applying CLAHE + Background Removal (NumPy Engine)...


Processing train/male:   0%|          | 0/665 [00:00<?, ?it/s]

Processing train/female:   0%|          | 0/665 [00:00<?, ?it/s]

Processing val/male:   0%|          | 0/167 [00:00<?, ?it/s]

Processing val/female:   0%|          | 0/167 [00:00<?, ?it/s]

✅ Preprocessing complete! Handing over to PyTorch...
🚀 Firing up PyTorch on CUDA for Version 1.3...
🗂️ Classes detected by PyTorch: {'female': 0, 'male': 1}
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 212MB/s] 



Epoch 1/10
---------------


Training:   0%|          | 0/42 [00:00<?, ?it/s]

AttributeError: Caught AttributeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 245, in __getitem__
    sample = self.loader(path)
             ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 284, in default_loader
    return pil_loader(path)
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 263, in pil_loader
    img = Image.open(f)
          ^^^^^^^^

# Training with the background removal

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
from PIL import Image
import requests

# --- THE FIX: CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    """Bypasses Pillow's broken JPEG parser by forcing OpenCV to read the images."""
    img = cv2.imread(path)
    if img is None:
        # Fallback to a blank image if something is corrupt, to avoid crashing the whole epoch
        return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# We are pointing directly to the ALREADY PROCESSED folder
processed_base_dir = '/kaggle/working/processed_dataset'
print(f"📂 Loading pre-processed images from: {processed_base_dir}")

# --- 2. HYPERPARAMETERS ---
BATCH_SIZE = 32
EPOCHS = 10  
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Version 1.3...")

# --- 3. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Injecting the custom_opencv_loader here!
train_dataset = datasets.ImageFolder(
    os.path.join(processed_base_dir, 'train'), 
    transform=train_transforms,
    loader=custom_opencv_loader
)
val_dataset = datasets.ImageFolder(
    os.path.join(processed_base_dir, 'val'), 
    transform=val_transforms,
    loader=custom_opencv_loader
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 4. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 VERSION 1.3 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 1.3
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_3_gender.pth')
print("💾 Saved Version 1.3 weights as 'swin_v1_3_gender.pth'")

📂 Loading pre-processed images from: /kaggle/working/processed_dataset
🚀 Firing up PyTorch on CUDA for Version 1.3...
🗂️ Classes detected by PyTorch: {'female': 0, 'male': 1}

Epoch 1/10
---------------


Training:   0%|          | 0/42 [00:00<?, ?it/s]

Testing:   0%|          | 0/11 [00:00<?, ?it/s]

Train Loss: 0.6740 | Train Acc: 0.5827
Val Loss:   0.6158 | Val Acc:   0.6347

Epoch 2/10
---------------


Training:   0%|          | 0/42 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/11 [00:00<?, ?it/s]

Train Loss: 0.2907 | Train Acc: 0.8481
Val Loss:   0.7238 | Val Acc:   0.7365

Epoch 10/10
---------------


Training:   0%|          | 0/42 [00:00<?, ?it/s]

Testing:   0%|          | 0/11 [00:00<?, ?it/s]

Train Loss: 0.2742 | Train Acc: 0.8722
Val Loss:   0.9144 | Val Acc:   0.7335

🎉 VERSION 1.3 TRAINING COMPLETE in 3m 34s
🏆 Best Validation Accuracy: 0.7365
💾 Saved Version 1.3 weights as 'swin_v1_3_gender.pth'


# Training without removing background, just CLAHE

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
from PIL import Image
import requests


# --- 1. THE NEW PREPROCESSING ENGINE (CLAHE + BRIGHTNESS, NO REMBG) ---
def enhance_contrast_brightness(input_path, output_path):
    try:
        img = cv2.imread(input_path)
        if img is None: return

        # 1. CLAHE (Contrast Enhancement)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l_channel, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l_channel)
        merged = cv2.merge((cl, a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

        # 2. Brightness Enhancement (alpha = contrast control, beta = brightness control)
        # Bumping brightness by 15 points, keeping contrast multiplier at 1.1
        final_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        cv2.imwrite(output_path, final_img)
    except Exception as e:
        print(f"Skipping corrupted image {input_path}: {e}")

raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/clean_v1_1_dataset'
processed_base_dir = '/kaggle/working/processed_dataset'

print("✨ Phase 1: Applying CLAHE + Brightness Enhancement...")
for split in ['train', 'val']:
    for gender in ['male', 'female']:
        in_folder = os.path.join(raw_base_dir, split, gender)
        out_folder = os.path.join(processed_base_dir, split, gender)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{gender}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_contrast_brightness(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 10  
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Version 1.5...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 5. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 VERSION 1.5 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 1.4
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_5_gender.pth')
print("💾 Saved Version 1.4 weights as 'swin_v1_5_gender.pth'")

# Version 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from PIL import Image
import requests

# --- 1. THE ULTIMATE V2.0 PREPROCESSING ENGINE ---
def enhance_v2_0_prep(input_path, output_path):
    """CLAHE + Brightness + Gaussian Blur + Letterbox Padding"""
    try:
        img = cv2.imread(input_path)
        if img is None: return

        # 1. CLAHE & Brightness
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        merged = cv2.merge((clahe.apply(l), a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        bright_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        # 2. Gaussian Blur (Kills digital noise/grain)
        blurred_img = cv2.GaussianBlur(bright_img, (3, 3), 0)

        # 3. Letterbox Padding (Stops PyTorch from squishing proportions)
        h, w = blurred_img.shape[:2]
        longest_edge = max(h, w)
        
        top = (longest_edge - h) // 2
        bottom = longest_edge - h - top
        left = (longest_edge - w) // 2
        right = longest_edge - w - left
        
        padded_img = cv2.copyMakeBorder(
            blurred_img, top, bottom, left, right, 
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )

        cv2.imwrite(output_path, padded_img)
    except Exception as e:
        print(f"Skipping corrupted image {input_path}: {e}")

# IMPORTANT: Update this path to your newly uploaded dataset!
raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/cleaned_dataset_v2_0/cleaned_dataset_v2_0' 
processed_base_dir = '/kaggle/working/processed_dataset_v2'

print("✨ Phase 1: Applying V2.0 SOTA Preprocessing...")
for split in ['train', 'val']:
    for gender in ['male', 'female']:
        in_folder = os.path.join(raw_base_dir, split, gender)
        out_folder = os.path.join(processed_base_dir, split, gender)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{gender}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_v2_0_prep(in_path, out_path)

print("✅ V2.0 Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 12  # Pushed to 12 since we have a massive 4,000 image training set
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Version 2.0...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 5. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
model.head = nn.Linear(in_features, 2) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 VERSION 2.0 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save Version 2.0
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v2_0_gender.pth')
print("💾 Saved Version 2.0 weights as 'swin_v2_0_gender.pth'")

✨ Phase 1: Applying V2.0 SOTA Preprocessing...


Processing train/male:   0%|          | 0/2000 [00:00<?, ?it/s]

Processing train/female:   0%|          | 0/2000 [00:00<?, ?it/s]

Processing val/male:   0%|          | 0/500 [00:00<?, ?it/s]

Processing val/female:   0%|          | 0/500 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/16 [00:00<?, ?it/s]

Train Loss: 0.0769 | Train Acc: 0.9693
Val Loss:   0.2186 | Val Acc:   0.9310

Epoch 12/12
---------------


Training:   0%|          | 0/63 [00:00<?, ?it/s]

Testing:   0%|          | 0/16 [00:00<?, ?it/s]

Train Loss: 0.0690 | Train Acc: 0.9748
Val Loss:   0.2331 | Val Acc:   0.9310

🎉 VERSION 2.0 TRAINING COMPLETE in 11m 20s
🏆 Best Validation Accuracy: 0.9310
💾 Saved Version 2.0 weights as 'swin_v2_0_gender.pth'


## Age Detection

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from PIL import Image
import requests

# --- 1. SOTA PREPROCESSING (Matched for Live Interface Parity) ---
def enhance_age_prep(input_path, output_path):
    """CLAHE + Brightness + Gaussian Blur + Letterbox Padding"""
    try:
        img = cv2.imread(input_path)
        if img is None: return

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        merged = cv2.merge((clahe.apply(l), a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        bright_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        blurred_img = cv2.GaussianBlur(bright_img, (3, 3), 0)

        h, w = blurred_img.shape[:2]
        longest_edge = max(h, w)
        top = (longest_edge - h) // 2
        bottom = longest_edge - h - top
        left = (longest_edge - w) // 2
        right = longest_edge - w - left
        
        padded_img = cv2.copyMakeBorder(
            blurred_img, top, bottom, left, right, 
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )

        cv2.imwrite(output_path, padded_img)
    except Exception as e:
        print(f"Skipping corrupted image {input_path}: {e}")

# IMPORTANT: Update this path to your newly uploaded AGE dataset!
raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/dataset_age_v1/dataset_age_v1' 
processed_base_dir = '/kaggle/working/processed_age_dataset'

print("✨ Phase 1: Applying Preprocessing to Age Dataset...")
# The 5 Age Buckets
AGE_GROUPS = ["kids", "teens", "young_adults", "middle_aged", "seniors"]

for split in ['train', 'val']:
    for group in AGE_GROUPS:
        in_folder = os.path.join(raw_base_dir, split, group)
        out_folder = os.path.join(processed_base_dir, split, group)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{group}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_age_prep(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 15  # Age is harder than Gender, giving it 15 epochs
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Age Detection V1.0...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 5. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
# THE FIX: We now have 5 output classes, not 2!
model.head = nn.Linear(in_features, 5) 
model = model.to(DEVICE)

# Perfectly balanced dataset = No dynamic weights needed
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 AGE VERSION 1.0 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save the Age Brain
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_0_age.pth')
print("💾 Saved weights as 'swin_v1_0_age.pth'")

✨ Phase 1: Applying Preprocessing to Age Dataset...


Processing train/kids:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/teens:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/young_adults:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/middle_aged:   0%|          | 0/944 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.3659 | Train Acc: 0.8646
Val Loss:   0.8002 | Val Acc:   0.7059

Epoch 15/15
---------------


Training:   0%|          | 0/74 [00:00<?, ?it/s]

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.3379 | Train Acc: 0.8706
Val Loss:   0.8026 | Val Acc:   0.7263

🎉 AGE VERSION 1.0 TRAINING COMPLETE in 16m 45s
🏆 Best Validation Accuracy: 0.7322
💾 Saved weights as 'swin_v1_0_age.pth'


### Applying Label Smoothing

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from PIL import Image
import requests

# --- 1. SOTA PREPROCESSING (Matched for Live Interface Parity) ---
def enhance_age_prep(input_path, output_path):
    """CLAHE + Brightness + Gaussian Blur + Letterbox Padding"""
    try:
        img = cv2.imread(input_path)
        if img is None: return

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        merged = cv2.merge((clahe.apply(l), a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        bright_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        blurred_img = cv2.GaussianBlur(bright_img, (3, 3), 0)

        h, w = blurred_img.shape[:2]
        longest_edge = max(h, w)
        top = (longest_edge - h) // 2
        bottom = longest_edge - h - top
        left = (longest_edge - w) // 2
        right = longest_edge - w - left
        
        padded_img = cv2.copyMakeBorder(
            blurred_img, top, bottom, left, right, 
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )

        cv2.imwrite(output_path, padded_img)
    except Exception as e:
        print(f"Skipping corrupted image {input_path}: {e}")

# IMPORTANT: Update this path to your newly uploaded AGE dataset!
raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/dataset_age_v1/dataset_age_v1' 
processed_base_dir = '/kaggle/working/processed_age_dataset'

print("✨ Phase 1: Applying Preprocessing to Age Dataset...")
# The 5 Age Buckets
AGE_GROUPS = ["kids", "teens", "young_adults", "middle_aged", "seniors"]

for split in ['train', 'val']:
    for group in AGE_GROUPS:
        in_folder = os.path.join(raw_base_dir, split, group)
        out_folder = os.path.join(processed_base_dir, split, group)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{group}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_age_prep(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 15  # Age is harder than Gender, giving it 15 epochs
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Age Detection V1.0...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 5. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
# THE FIX: We now have 5 output classes, not 2!
model.head = nn.Linear(in_features, 5) 
model = model.to(DEVICE)

# Perfectly balanced dataset = No dynamic weights needed
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 AGE VERSION 1.0 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save the Age Brain
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_2_age.pth')
print("💾 Saved weights as 'swin_v1_2_age.pth'")